# Notebook 05 — Risk Assessment Agent (Logistic Regression)
Trains a calibrated Logistic Regression on the unified feature matrix. Produces risk_score ∈ [0,1].

In [ ]:
import os, sys, json, pickle, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              classification_report, RocCurveDisplay)
from sklearn.model_selection import GridSearchCV

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

COLAB_BASE = "/content/drive/MyDrive/PAS"
if os.path.exists(COLAB_BASE):
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    BASE = COLAB_BASE
else:
    BASE = str(Path(".").resolve().parent)

CACHE_DIR  = os.path.join(BASE, "EXPERIMENT_CACHE")
MODELS_DIR = os.path.join(BASE, "MODELS"); os.makedirs(MODELS_DIR, exist_ok=True)
RESULTS_DIR = os.path.join(BASE, "RESULTS", "metrics"); os.makedirs(RESULTS_DIR, exist_ok=True)
print("Setup OK")

In [ ]:
features_path = os.path.join(CACHE_DIR, "features.parquet")
col_path = os.path.join(MODELS_DIR, "feature_columns.json")

assert os.path.exists(features_path), f"Run Notebook 04 first! Not found: {features_path}"
assert os.path.exists(col_path), f"feature_columns.json not found: {col_path}"

df = pd.read_parquet(features_path)
with open(col_path) as f:
    FEATURE_COLS = json.load(f)["columns"]

train_df = df[df["split"] == "train"]
val_df   = df[df["split"] == "val"]
test_df  = df[df["split"] == "test"]

X_train = train_df[FEATURE_COLS].values
y_train = train_df["label_int"].values
X_val   = val_df[FEATURE_COLS].values
y_val   = val_df["label_int"].values
X_test  = test_df[FEATURE_COLS].values
y_test  = test_df["label_int"].values

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"Train malicious rate: {y_train.mean():.2%}")

In [ ]:
# IMPORTANT: Fit scaler ONLY on training data (no data leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)      # transform only
X_test_scaled  = scaler.transform(X_test)     # transform only
print("Scaler fit on train only (no leakage).")

# Hyperparameter tuning via cross-validation on training set
param_grid = {"C": [0.01, 0.1, 1.0, 10.0]}
lr_base = LogisticRegression(solver="saga", max_iter=1000, random_state=RANDOM_SEED,
                              class_weight="balanced")
grid = GridSearchCV(lr_base, param_grid, cv=3, scoring="f1_macro", n_jobs=-1, verbose=1)
grid.fit(X_train_scaled, y_train)

best_C = grid.best_params_["C"]
print(f"Best C: {best_C} (val F1-macro: {grid.best_score_:.4f})")

# Calibrate with Platt scaling
lr_best = LogisticRegression(C=best_C, solver="saga", max_iter=1000,
                             random_state=RANDOM_SEED, class_weight="balanced")
calibrated_lr = CalibratedClassifierCV(lr_best, cv=5, method="sigmoid")
calibrated_lr.fit(X_train_scaled, y_train)
print("Calibrated model trained.")

In [ ]:
# Evaluate on validation (for threshold tuning) and test (for paper)
for split_name, X_s, y_s in [("val", X_val_scaled, y_val), ("test", X_test_scaled, y_test)]:
    y_pred  = calibrated_lr.predict(X_s)
    y_proba = calibrated_lr.predict_proba(X_s)[:, 1]
    fnr = ((y_s == 1) & (y_pred == 0)).sum() / max(1, (y_s == 1).sum())
    print(f"\n{'='*50} {split_name.upper()} {'='*50}")
    print(classification_report(y_s, y_pred, target_names=["benign", "malicious"]))
    print(f"ROC-AUC: {roc_auc_score(y_s, y_proba):.4f} | FNR: {fnr:.4f}")

# Feature importance
coefs = calibrated_lr.calibrated_classifiers_[0].estimator.coef_[0]
importance = sorted(zip(FEATURE_COLS, np.abs(coefs)), key=lambda x: x[1], reverse=True)
fig, ax = plt.subplots(figsize=(10, 5))
names, vals = zip(*importance)
ax.barh(names[::-1], vals[::-1], color="#3498db")
ax.set_title("Risk Model Feature Importance (|coefficient|)", fontweight="bold")
ax.set_xlabel("|Coefficient|")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "05_feature_importance.png"), bbox_inches="tight", dpi=150)
plt.show()

# Save model, scaler, columns
with open(os.path.join(MODELS_DIR, "logistic_regression.pkl"), "wb") as f:
    pickle.dump(calibrated_lr, f)
with open(os.path.join(MODELS_DIR, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)
print("Models saved.")